# Notebook 02: Webの仕組み

SQLインジェクションやXSSを理解するには、まず **Webがどうやって動くのか** を知る必要があります。
攻撃者はWebの仕組みの「隙間」を突くからです。

## 学習目標
1. HTTP リクエスト・レスポンスの構造を理解する
2. GET と POST の違いを知る
3. Cookie とセッションの仕組みを理解する
4. ブラウザがHTMLをどう処理するかを知る（XSSの前提）

## 1. HTTP：Webの共通言語

あなたがブラウザで `https://example.com/products?q=リンゴ` を開くとき、
舞台裏では次のような「会話」が起きています。

### リクエスト（ブラウザ → サーバー）
```
GET /products?q=リンゴ HTTP/1.1
Host: example.com
User-Agent: Mozilla/5.0 ...
Cookie: session_id=abc123
```

### レスポンス（サーバー → ブラウザ）
```
HTTP/1.1 200 OK
Content-Type: text/html
Set-Cookie: session_id=abc123

<html>...<p>リンゴ: 100円</p>...</html>
```

### HTTPの重要な特徴
- **テキストベース**: すべてが読める文字列
- **ステートレス**: サーバーはリクエストを「覚えない」（だからCookieが必要）
- **クライアントが信頼されない**: サーバーは受け取ったデータを検証すべき

↑ この最後の原則が破られると、多くの脆弱性が生まれます！

In [ ]:
# Python で HTTP リクエストを送る（requests ライブラリ）
import requests

# GET リクエスト
response = requests.get("https://httpbin.org/get", params={"name": "alice", "age": 25})

print("=== リクエスト ===")
print(f"URL: {response.url}")
print(f"メソッド: GET")
print()
print("=== レスポンス ===")
print(f"ステータスコード: {response.status_code}")
print(f"Content-Type: {response.headers['Content-Type']}")
print()
data = response.json()
print("サーバーが受け取ったパラメータ:")
print(data["args"])

In [ ]:
# POST リクエスト（ログインフォームのシミュレーション）
response = requests.post(
    "https://httpbin.org/post",
    data={"username": "alice", "password": "secret123"}
)

data = response.json()
print("=== POST で送ったデータ ===")
print(data["form"])
print()
print("注意: GETと違ってURLにパスワードが見えない")
print(f"URL: {response.url}")
print()
print("ただし POST も暗号化（HTTPS）なしではネットワーク上で見える！")

## 2. GETとPOSTの使い分け

| | GET | POST |
|-|-----|------|
| データの場所 | URL（クエリストリング） | リクエストボディ |
| 用途 | データ取得・検索 | データ送信・更新 |
| ブックマーク | ✅ できる | ❌ できない |
| 履歴・ログ | ✅ URLが残る | 🟡 ボディは残りにくい |
| パスワード送信 | ❌ 絶対ダメ（URLに残る） | ✅（ただしHTTPS必須） |

### セキュリティ上の問題
```
❌ 悪い例：
GET /login?username=alice&password=secret123

→ ブラウザ履歴、サーバーログ、プロキシのログにパスワードが残る！
```

## 3. Cookie とセッション

HTTPはステートレスなので、「あなたが誰か」をサーバーは忘れます。
これを解決するのが **Cookie** です。

### ログインの流れ
```
1. ブラウザ → サーバー: POST /login (username=alice, password=...)
2. サーバー: パスワード確認 → OK
3. サーバー → ブラウザ: Set-Cookie: session_id=xyz789 (レスポンスヘッダー)
4. ブラウザ: Cookieを保存
5. 次のリクエストから自動的に Cookie: session_id=xyz789 を送る
6. サーバー: session_id でユーザーを識別する
```

### Cookieの属性（セキュリティ重要！）

```
Set-Cookie: session_id=xyz789; HttpOnly; Secure; SameSite=Strict
```

| 属性 | 意味 | 欠如した場合の危険 |
|-----|------|------------------|
| `HttpOnly` | JavaScriptからアクセス不可 | XSSでCookie盗まれる |
| `Secure` | HTTPSのみで送信 | 通信傍受で盗まれる |
| `SameSite=Strict` | 他サイトからのリクエストに含めない | CSRF攻撃を受ける |

In [ ]:
# Cookie の仕組みをシミュレーション
import requests

# セッションオブジェクトはCookieを自動で管理する
session = requests.Session()

# Cookieをセットしてくれるエンドポイントに接続
r1 = session.get("https://httpbin.org/cookies/set", params={"user": "alice", "role": "admin"})
print("セット後のCookie:")
print(session.cookies.get_dict())
print()

# 次のリクエストでは自動的にCookieが送られる
r2 = session.get("https://httpbin.org/cookies")
print("サーバーが受け取ったCookie:")
print(r2.json()["cookies"])
print()
print("→ サーバーはCookieで user=alice, role=admin と識別できる")

## 4. ブラウザとHTMLの処理

ブラウザはサーバーから受け取ったHTMLを「解釈」して画面に表示します。
ここが **XSS（クロスサイトスクリプティング）** の舞台になります。

### HTMLの中でJavaScriptは実行される

```html
<!-- 通常のHTMLページ -->
<p>こんにちは、alice さん！</p>

<!-- もし攻撃者が「alice さん」の部分に以下を注入できたら？ -->
<p>こんにちは、<script>alert('あなたのCookieは: ' + document.cookie)</script> さん！</p>
```

ブラウザは `<script>` タグを見つけると、その中のJavaScriptを **実行** します。
これがXSSの核心です。Notebook 04 で詳しく扱います。

### なぜ危険か
- `document.cookie` でCookieにアクセスできる
- Cookieが盗まれると **セッションハイジャック**（他人のアカウントを乗っ取る）が可能

In [ ]:
# HTMLエスケープの重要性
# ユーザー入力を安全に表示する方法
from html import escape

# 攻撃者が入力する可能性のある文字列
user_input = "<script>alert('XSS!')</script>"

print("=== エスケープなし（危険！） ===")
dangerous_html = f"<p>こんにちは、{user_input} さん！</p>"
print(dangerous_html)
print("→ ブラウザはscriptタグを実行してしまう")
print()

print("=== エスケープあり（安全） ===")
safe_html = f"<p>こんにちは、{escape(user_input)} さん！</p>"
print(safe_html)
print("→ ブラウザはタグとして解釈せず、文字として表示する")
print()

print("HTMLエスケープの変換ルール:")
chars = ['<', '>', '&', '"', "'"]
for c in chars:
    print(f"  {c!r} → {escape(c)!r}")

## 5. HTTPステータスコード一覧

レスポンスのステータスコードはセキュリティテストで重要な手がかりになります。

| コード | 意味 | セキュリティ的な意味 |
|--------|------|--------------------|
| 200 OK | 成功 | リソースが存在・アクセス可能 |
| 301/302 | リダイレクト | オープンリダイレクト脆弱性の可能性 |
| 401 Unauthorized | 認証が必要 | ログインが必要 |
| 403 Forbidden | アクセス禁止 | 認証済みだが権限不足 |
| 404 Not Found | 見つからない | リソースの存在確認に使われる |
| 500 Internal Server Error | サーバーエラー | エラーメッセージに情報が含まれる可能性 |

In [ ]:
# ステータスコードの確認
import requests

test_urls = [
    "https://httpbin.org/status/200",
    "https://httpbin.org/status/404",
    "https://httpbin.org/status/403",
    "https://httpbin.org/status/500",
]

for url in test_urls:
    try:
        r = requests.get(url, timeout=5)
        print(f"{url.split('/')[-1]} → {r.status_code} {r.reason}")
    except Exception as e:
        print(f"エラー: {e}")

## まとめ

- HTTP は **テキストベース** の通信プロトコル。全て「読める」
- **GET** はURLにデータが見える。パスワードには使わない
- **Cookie** はセッション管理に使う。`HttpOnly`, `Secure` 属性が重要
- ユーザー入力を HTML に埋め込むときは **エスケープ** が必須（XSS対策）

次のノートブックでは、いよいよ最も有名な脆弱性 **SQLインジェクション** を学びます！

## 演習

1. ブラウザの開発者ツール（F12）を開いて、NetworkタブでHTTPリクエストを観察してみよう
2. `requests` を使って `https://httpbin.org/headers` にアクセスし、どんなヘッダーが送られているか確認しよう
3. `html.escape()` を使って以下の文字列を安全にエスケープしよう：
   - `"<img src=x onerror=alert(1)>"`
   - `"'; DROP TABLE users; --"`